# OpenAlex corpus

## Setup

In [ ]:
from pathlib import Path
import os

import duckdb
import pandas as pd
import requests
import yaml
from dotenv import load_dotenv

def get_config(path):
    path = Path(path)
    with path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)
    return config

config = get_config("../config/config.yaml")
corpus = config["corpus"]

load_dotenv()
config["api_key"] = os.environ["OPENALEX_API_KEY"]

## Query

In [ ]:
works_url = "https://api.openalex.org/works"

subfields = "|".join(str(subfield) for subfield in corpus["subfields"])

filters = ",".join([
    f"publication_year:>{corpus['min_year'] - 1}",  # filter is strict >, so -1 keeps min_year in
    f"has_abstract:{str(corpus['has_abstract']).lower()}",
    f"primary_topic.subfield.id:{subfields}",
])

params = {
    "filter": filters,
    "select": ",".join(corpus["cols_to_select"]),
    "api_key": config["api_key"],
}

## Helpers

In [ ]:
def paginate(url, params=None, per_page=200):
    """Yield successive pages from a cursor-paginated OpenAlex endpoint."""
    params = dict(params or {})
    params["per-page"] = per_page
    cursor = "*"

    while cursor:
        params["cursor"] = cursor
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()

        if not page.get("results"):
            break

        yield page
        cursor = page.get("meta", {}).get("next_cursor")

In [ ]:
def reconstruct_abstract(inverted_abstract):
    if not inverted_abstract:
        return None, 0

    abstract = []
    for word, positions in inverted_abstract.items():
        for position in positions:
            while len(abstract) <= position:
                abstract.append(None)
            abstract[position] = word

    n_gaps = abstract.count(None)
    text = " ".join([word for word in abstract if word is not None])
    return text, n_gaps

In [ ]:
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [2]}) == ("The cat sat", 0)
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [3]}) == ("The cat sat", 1)
assert reconstruct_abstract(None) == (None, 0)
assert reconstruct_abstract({}) == (None, 0)

In [ ]:
def remove_prefix(item):
    if item is None:
        return None

    value = item.removeprefix("https://doi.org/")
    value = value.removeprefix("https://openalex.org/subfields/")
    value = value.removeprefix("https://openalex.org/")

    return value

In [ ]:
def flatten_work(work):
    work_id = remove_prefix(work['id'])
    abstract, gap_count = reconstruct_abstract(work['abstract_inverted_index'])

    work_row = {'work_id': work_id,
                'doi': remove_prefix(work['doi']),
                'title': work['title'],
                'abstract': abstract,
                'abstract_gap_count': gap_count,
                'publication_year': work['publication_year'],
                'publication_date': work['publication_date'],
                'cited_by_count': work['cited_by_count'],
                'subfield_id': int(remove_prefix(work['primary_topic']['subfield']['id'])),
                'subfield_name': work['primary_topic']['subfield']['display_name'],
                'topic_id': remove_prefix(work['primary_topic']['id']),
                'topic_name': work['primary_topic']['display_name']
                }

    author_rows = []
    for author_order, authorship in enumerate(work['authorships'], start=1):
        author_row = {'work_id': work_id,
                      'author_order': author_order,
                      'author_id': remove_prefix(authorship['author']['id']),
                      'author_name': authorship['author']['display_name']
                      }
        author_rows.append(author_row)

    reference_rows = []
    for url in work['referenced_works']:
        reference_row = {'citing_work_id': work_id, 'cited_work_id': remove_prefix(url)}
        reference_rows.append(reference_row)

    return (work_row, author_rows, reference_rows)

## Sanity checks on one page

In [ ]:
pages = paginate(works_url, params, per_page=25)
first_page = next(pages)
print("Number of results:", first_page["meta"]["count"])

In [ ]:
for work in first_page["results"]:
    print(work["title"])

In [ ]:
print(first_page["results"][0].keys())

In [ ]:
[reconstruct_abstract(work["abstract_inverted_index"]) for work in first_page["results"][:5]]
# All abstracts read normally, seems to work okay

## Database

In [ ]:
con = duckdb.connect(corpus["db_path"])

In [ ]:
con.execute("""CREATE TABLE IF NOT EXISTS works (
                    work_id VARCHAR,
                    doi VARCHAR,
                    title VARCHAR,
                    abstract VARCHAR,
                    abstract_gap_count INTEGER,
                    publication_year INTEGER,
                    publication_date DATE,
                    cited_by_count INTEGER,
                    subfield_id INTEGER,
                    subfield_name VARCHAR,
                    topic_id VARCHAR,
                    topic_name VARCHAR,
                    PRIMARY KEY (work_id)
                    )""")

con.execute("""CREATE TABLE IF NOT EXISTS work_authors (
                    work_id VARCHAR,
                    author_order INTEGER,
                    author_id VARCHAR,
                    author_name VARCHAR,
                    PRIMARY KEY (work_id, author_order)
                    )""")

con.execute("""CREATE TABLE IF NOT EXISTS work_references (
                    citing_work_id VARCHAR,
                    cited_work_id VARCHAR
                    )""")

In [ ]:
print(con.sql("SHOW TABLES"))
print(con.sql("DESCRIBE works"))
print(con.sql("DESCRIBE work_authors"))
print(con.sql("DESCRIBE work_references"))

## Check flatten_work against the tables

In [ ]:
work_columns = set(con.sql("DESCRIBE works").df()["column_name"])
author_columns = set(con.sql("DESCRIBE work_authors").df()["column_name"])
reference_columns = set(con.sql("DESCRIBE work_references").df()["column_name"])

n_authors = 0
n_references = 0
n_null_author_ids = 0

for work in first_page["results"]:
    work_row, author_rows, reference_rows = flatten_work(work)
    work_id = work_row["work_id"]

    # The work row has exactly the columns of the works table
    assert set(work_row) == work_columns, work_id

    # One author row per authorship, numbered from 1, all pointing back to this work
    assert len(author_rows) == len(work["authorships"]), work_id
    for row in author_rows:
        assert set(row) == author_columns, work_id
        assert row["work_id"] == work_id, work_id
    if author_rows:
        assert author_rows[0]["author_order"] == 1, work_id

    # One reference row per referenced work, all pointing back to this work
    assert len(reference_rows) == len(work["referenced_works"]), work_id
    for row in reference_rows:
        assert set(row) == reference_columns, work_id
        assert row["citing_work_id"] == work_id, work_id

    n_authors += len(author_rows)
    n_references += len(reference_rows)
    n_null_author_ids += sum(row["author_id"] is None for row in author_rows)

print(f"All checks passed: {len(first_page['results'])} works, {n_authors} author rows, "
      f"{n_references} reference rows, {n_null_author_ids} author rows with no author ID")